In [ ]:
# Реализуйте пакетный градиентный спуск (**Batch Gradient Descent**) с ранней остановкой (**Early Stopping**) 
# для Softmax Regression без использования Scikit-Learn, только с помощью NumPy. 
# Используйте его для задачи классификации, например, на датасете Iris.

In [1]:
# Detect iris virginica
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
print(list(iris), "\n")
print(iris.data.head(3),  "\n")
# instances are not shuffled
print(iris.target.head(3),  "\n")
print(iris.target_names,  "\n")
# чашелистик, лепесток

['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'] 

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2 

0    0
1    0
2    0
Name: target, dtype: int32 

['setosa' 'versicolor' 'virginica'] 



In [19]:
# Detect iris virginica
import numpy as np
import matplotlib.pyplot as plt

X = iris.data.values
y = iris.target
# split data 80:20
def split_data(x, y):
    # X_train, X_test, y_train, y_test
    train_end = int(len(X) / 100 * 80)
    valid_start = train_end+1
    X_train = x[0:train_end]
    y_train = x[0:train_end]
    X_test = x[valid_start:len(X)]
    y_test = y[valid_start:len(X)]
    return X_train,y_train, X_test, y_test

X_train, y_train, X_test, y_test = split_data(X,y)
print(len(X_train))
print(len(y_train))
print(len(X_test))
print(len(y_test))

# Фильтруем и берем по каждому классу y к x значению
setosa_test = X[y == 0][0]
versicolor_test = X[y == 1][0]
virginica_test = X[y == 2][0]
print(setosa_test)
print(versicolor_test)
print(virginica_test)


120
120
29
29
[5.1 3.5 1.4 0.2]
[7.  3.2 4.7 1.4]
[6.3 3.3 6.  2.5]


In [ ]:
# Batch gradient
# Early stopping
# SoftmaxRegression


import numpy as np
import math

np.random.seed(42)

def theta_init(X):
    # Independent variables (X) — площадь и количество комнат. Это то, что мы подаём на вход модели.
    # Dependent variable (y) — цена. Это то, что мы хотим предсказать, и она зависит от входных признаков.
    # Количество элементов в theta = количество признаков + 1 (для свободного члена theta).
    # X = [ [50, 2], [2,3], [4,4]    # 2 признака]
    n_features = len(X[0])
    n_classes = 3 # Так как 3 типа цветков
    theta = np.random.randn(n_features, n_classes) * 0.01
    # shape (4, 3)
    return theta



class Logistics_Regression:
    def __init__(self, lr, iterations, withEarlyStopping, EARLY_STOP_MAX_I = 30):
        # lr learning rate
        self.lr = lr
        self.iterations = iterations
        self.withEarlyStopping = withEarlyStopping
        self.minError = None 
        self.bestTheta = None
        self.EARLY_STOP_MAX_I = EARLY_STOP_MAX_I
        self.earlyStopIteration = 0
        self.cost_lst = []

    def _score(self, x, theta):
        scores = []
        for _, x_val in enumerate(x):
            row = []
            for theta_k in theta.T:
                row.append(theta_k @ x_val)
            scores.append(row)
        return np.array(scores)

    def _softmax(self, S):
        P = []
        for score in S:
            sum_exp = 0
            for value in score:			
                sum_exp += math.exp(value)
            row = [] 
            for value in score:
                row.append(math.exp(value) /  sum_exp)
            P.append(row)
        return np.array(P)
        

    def fit(self, X, y):
        theta = theta_init(X)
        m = len(X)
        vectorX = X
        
        for i in range(self.iterations):
            # TODO разобрать
            S = self._score(vectorX, theta)
            P = self._softmax(S)

            grad = vectorX.T @ (P - y) / m
            cost_value = -np.sum(y * np.log(P)) / m 
            theta = theta -  self.lr * grad
            rounded_cost_value = round(cost_value,5)
            self.cost_lst.append(cost_value)

            if self.minError is None or rounded_cost_value < self.minError:
                self.minError = rounded_cost_value
                self.bestTheta = theta
                self.earlyStopIteration = 0
            else:
                self.earlyStopIteration += 1
                if self.withEarlyStopping and self.earlyStopIteration > self.EARLY_STOP_MAX_I:
                    print("stopped on", i)
                    break
        return theta



    def predict(self, X):
        S = self._score(X, self.bestTheta)
        # argmax return a class
        return S, np.argmax(S, axis=1)

Y_onehot = np.eye(3)[np.array(y)]

# classifier = Logistics_Regression(0.2, 500, True) , True - с early stopping
# Но у меня там с точностью проблемы возникают
classifier = Logistics_Regression(0.2, 500, False)
classifier.fit(X,Y_onehot)
setosa_pred_S, setosa_pred_R = classifier.predict([setosa_test])
versicolor_pred_S, versicolor_pred_R = classifier.predict([versicolor_test])
virginica_pred_S, virginica_pred_R = classifier.predict([virginica_test]) 
print("Predict is setosa:", setosa_pred_S,setosa_pred_R ) # should be 0
print("Predict is versicolor:", versicolor_pred_S, versicolor_pred_R ) # should be 1
print("Predict is virginica:",virginica_pred_S, virginica_pred_R ) # should be 2



Predict is setosa: [[  9.19887964   4.20955661 -13.29470006]] [0]
Predict is versicolor: [[-1.69835152  3.43566275 -1.55031496]] [1]
Predict is virginica: [[-7.95574471  0.63195671  7.52494683]] [2]
